### Paedawen: Training data generation

Set up environment (including `assume --env` credentials if required)

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from datagen import BedrockBatchGenerator, TrainingDataUploader
from paedacuteschema.prompt_builder import PromptBuilder
from paedacuteschema import Schema

In [3]:
pb = PromptBuilder()

#### Download documents from batch

In [4]:
gen = BedrockBatchGenerator(
    system_prompt=pb.build_datagen_prompt(),
    user_prompt_function=lambda doc: doc['content'],
    schema=Schema,
    schema_name="paedacuteschema",
    model_name="sonnet4",
    document_batches=["paedacute-batch-2026-06-11-001.tar.gz", "normal-batch-2025-12-31-001.tar.gz", "noncancer-batch-2025-12-31-001.tar.gz"],
)

In [5]:
gen.get_document_files_count()

3076

#### Sanity check

##### Start batch generation

In [6]:
gen.generate_via_batch(100, os.environ["BUCKET"],os.environ["BEDROCK_EXECUTION_ROLE"])

'datagen/2026-06-18-1510'

##### Download and parse batch outputs

In [7]:
gen.extract_batch_output(os.environ["BUCKET"])

LLM returned JSON: {
  "is_clinical_document": true,
  "extraction_reasoning": "(1) AMBIGUITIES: This is an anaphylaxis case with documented signs. 'Dx Anaphylaxis' is a diagnosis, not a sign in the schema. The wheeze is documented both as 'wheezing loudly' and 'audible wheeze', and auscultation confirms 'widespread wheeze'. Lip/facial swelling and urticaria are not AcuteSignType members so omitted. RR 36, HR 140, BP 88/52, SpO2 90% are vital signs/observations, not documented signs, so cannot be used to infer sig
LLM returned JSON: {
  "is_clinical_document": true,
  "extraction_reasoning": "(1) AMBIGUITIES: 'projectile vomiting' and 'forceful vomiting' are not AcuteSignType members, so omitted. 'sunken fontanelle' is not a member (only BULGING_FONTANELLE exists), so omitted. 'dry mucous membranes', 'visible peristalsis', 'palpable olive mass', and 'clinically dehydrated' are examination findings but not AcuteSignType members, so omitted. 'refusing the bottle'/'taking less than 20ml p

(100, 0)

#### Start datagen

In [ ]:
gen.generate_via_batch(3000, os.environ["BEDROCK_EXECUTION_ROLE"], os.environ["BUCKET"])

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

#### Upload formatted document:schema pairs as training data

In [ ]:
s3_uri = TrainingDataUploader.upload(
    schema=Schema,
    schema_name="paedacuteschema",
    system_prompt=pb.build_main_prompt(),
    short_description="paedawen-batch",
    long_description="Training data for Paedawen",
    input_folder=Path("./data/trainingdata"),
)